# E-Commerce Data Analysis Using Python
### All 15 Questions with Expected Outputs

In [19]:
import pandas as pd
import numpy as np

# Load all datasets
customers   = pd.read_csv('customers.csv')
orders      = pd.read_csv('orders.csv')
order_items = pd.read_csv('order_items.csv')
products    = pd.read_csv('products.csv')
payments    = pd.read_csv('payments.csv')
sellers     = pd.read_csv('sellers.csv')
geolocation = pd.read_csv('geolocation.csv')

# Rename products column to match expected name
products.rename(columns={'product category': 'product_category_name'}, inplace=True)

print('Data loaded successfully!')

Data loaded successfully!


## Q1. LIST ALL UNIQUE CITIES WHERE CUSTOMERS ARE LOCATED

In [20]:
customers['customer_city'].unique()

<StringArray>
[               'franca', 'sao bernardo do campo',             'sao paulo',
       'mogi das cruzes',              'campinas',        'jaragua do sul',
               'timoteo',              'curitiba',        'belo horizonte',
         'montes claros',
 ...
              'balbinos',          'serra bonita',          'venda branca',
           'sanga puita',               'queiroz',                'siriji',
   'natividade da serra',          'monte bonito',            'sao rafael',
     'eugenio de castro']
Length: 4119, dtype: str

## Q2. COUNT THE NUMBER OF ORDERS PLACED IN 2017

In [21]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

orders_2017 = orders[orders['order_purchase_timestamp'].dt.year == 2017]['order_id'].nunique()

orders_2017

45101

## Q3. FIND THE TOTAL SALES PER CATEGORY

In [22]:
temp = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='inner'
)
temp['product_category_name'] = temp['product_category_name'].fillna('Unknown')
category_sales = temp.groupby('product_category_name')['price'].sum().reset_index()
category_sales.rename(columns={
    'product_category_name': 'product_category_name',
    'price': 'total_sales'
}, inplace=True)

category_sales = category_sales.sort_values(
    by=['total_sales', 'product_category_name'],
    ascending=[False, True]
)

top9 = category_sales.head(9)
top9 = top9.reset_index(drop=True)
top9

,product_category_name,total_sales
0,HEALTH BEAUTY,1258681.34
1,Watches present,1205005.68
2,bed table bath,1036988.68
3,sport leisure,988048.97
4,computer accessories,911954.32
5,Furniture Decoration,729762.49
6,Cool Stuff,635290.85
7,housewares,632248.66
8,automotive,592720.11


## Q4. CALCULATE THE PERCENTAGE OF ORDERS THAT WERE PAID IN INSTALLMENTS

In [23]:
# STEP 1: Total unique orders
total_orders = payments['order_id'].nunique()

# STEP 2: Orders with installments > 1
installment_orders = payments[payments['payment_installments'] > 1]['order_id'].nunique()

# STEP 3: Percentage calculation
percentage = (installment_orders / total_orders) * 100

# STEP 4: Round to match SQL
percentage = round(percentage, 2)

percentage

51.46

## Q5. COUNT THE NUMBER OF CUSTOMERS FROM EACH STATE

In [24]:
customers_state = customers['customer_state'].value_counts()

customers_state

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

## Q6. CALCULATE THE NUMBER OF ORDERS PER MONTH IN 2018

In [25]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

orders_2018 = orders[orders['order_purchase_timestamp'].dt.year == 2018]

orders_count = orders_2018.groupby('year_month')['order_id'].count().reset_index()
orders_count.rename(columns={'order_id': 'no_of_orders'}, inplace=True)

orders_count = orders_count.sort_values('year_month')

orders_count = orders_count.reset_index(drop=True)

orders_count

,year_month,no_of_orders
0,2018-01,7269
1,2018-02,6728
2,2018-03,7211
3,2018-04,6939
4,2018-05,6873
5,2018-06,6167
6,2018-07,6292
7,2018-08,6512
8,2018-09,16
9,2018-10,4


## Q7. FIND THE AVERAGE NUMBER OF PRODUCTS PER ORDER, GROUPED BY CUSTOMER CITY

In [26]:
products_per_order = order_items.groupby('order_id').size().reset_index(name='product_count')
merged = products_per_order.merge(
    orders[['order_id', 'customer_id']],
    on='order_id'
).merge(
    customers[['customer_id', 'customer_city']],
    on='customer_id'
)

result = merged.groupby('customer_city')['product_count'].mean().reset_index()

result['avg_products_per_order'] = result['product_count'].round(2)

result = result[['customer_city', 'avg_products_per_order']]
result = result.sort_values(
    by=['avg_products_per_order', 'customer_city'],
    ascending=[False, True]
)
result.head(10)

,customer_city,avg_products_per_order
2619,padre carvalho,7.0
907,celso ramos,6.5
756,candido godoi,6.0
1154,datas,6.0
2264,matias olimpio,5.0
955,cidelandia,4.0
1146,curralinho,4.0
2395,morro de sao paulo,4.0
2817,picarra,4.0
3821,teixeira soares,4.0


## Q8. CALCULATE THE PERCENTAGE OF TOTAL REVENUE CONTRIBUTED BY EACH PRODUCT CATEGORY

In [35]:
temp = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='inner'
)
category_revenue = temp.groupby('product_category_name')['price'].sum().reset_index()
category_revenue.rename(columns={
    'product_category_name': 'product_category_name',
    'price': 'revenue_category'
}, inplace=True)
total_revenue = category_revenue['revenue_category'].sum()

category_revenue['revenue_percentage'] = (
    category_revenue['revenue_category'] / total_revenue
) * 100

category_revenue['revenue_category'] = category_revenue['revenue_category'].round(6)
category_revenue['revenue_percentage'] = category_revenue['revenue_percentage'].round(2)

category_revenue = category_revenue.sort_values(
    by=['revenue_category', 'product_category_name'],
    ascending=[False, True]
)

category_revenue = category_revenue.reset_index(drop=True)
category_revenue

,product_category_name,revenue_category,revenue_percentage
0,HEALTH BEAUTY,1258681.34,9.38
1,Watches present,1205005.68,8.98
2,bed table bath,1036988.68,7.73
3,sport leisure,988048.97,7.37
4,computer accessories,911954.32,6.80
...,...,...,...
68,flowers,1110.04,0.01
69,House Comfort 2,760.27,0.01
70,cds music dvds,730.00,0.01
71,Fashion Children's Clothing,569.85,0.00


## Q9. IDENTIFY THE CORRELATION BETWEEN PRODUCT PRICE AND THE NUMBER OF TIMES A PRODUCT HAS BEEN PURCHASED

In [28]:
# Frequency
product_freq = order_items.groupby('product_id').size().reset_index(name='purchase_count')

# Avg price
product_price = order_items.groupby('product_id')['price'].mean().reset_index()

# Merge
merged = product_freq.merge(product_price, on='product_id')

# Correlation
correlation = merged['price'].corr(merged['purchase_count'])

correlation

np.float64(-0.032139862680945167)

## Q10. CALCULATE THE TOTAL REVENUE GENERATED BY EACH SELLER, AND RANK THEM BY REVENUE

In [29]:
seller_revenue = order_items.groupby('seller_id')['price'].sum().reset_index()
seller_revenue.rename(columns={'price': 'revenue'}, inplace=True)

seller_revenue['revenue'] = seller_revenue['revenue'].round(6)

seller_revenue = seller_revenue.sort_values(
    by=['revenue', 'seller_id'],
    ascending=[False, True]
)

seller_revenue['rnk'] = seller_revenue['revenue'].rank(
    method='dense',
    ascending=False
).astype(int)

seller_revenue = seller_revenue.reset_index(drop=True)

seller_revenue.head(10)

,seller_id,revenue,rnk
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,2
2,4a3ca9315b744ce9f8e9374361493884,200472.92,3
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,4
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,5
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,6
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,7
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,8
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,9
9,955fee9216a65b617aa5c0531780ce60,135171.70,10


## Q11. CALCULATE THE MOVING AVERAGE OF ORDER VALUES FOR EACH CUSTOMER OVER THEIR ORDER HISTORY

In [30]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

order_value = order_items.groupby('order_id')['price'].sum().reset_index()
order_value.rename(columns={'price': 'order_value'}, inplace=True)

order_value = order_value.merge(
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
    on='order_id',
    how='inner'
)

order_value = order_value.sort_values(['customer_id', 'order_purchase_timestamp'])

order_value['moving_avg'] = order_value.groupby('customer_id')['order_value'] \
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())

order_value = order_value[[
    'customer_id',
    'order_id',
    'order_purchase_timestamp',
    'order_value',
    'moving_avg'
]]

order_value = order_value.reset_index(drop=True)
order_value.head(10)

,customer_id,order_id,order_purchase_timestamp,order_value,moving_avg
0,00012a2ce6f8dcda20d059ce98491703,5f79b5b0931d63f1a42989eb65b9da6e,2017-11-14 16:08:26,89.80,89.80
1,000161a058600d5901f007fab4c27140,a44895d095d7e0702b6a162fa2dbeced,2017-07-16 09:40:32,54.90,54.90
2,0001fd6190edaaf884bcaf3d49edf079,316a104623542e4d75189bb372bc5f8d,2017-02-28 11:06:43,179.99,179.99
3,0002414f95344307404f0ace7a26f1d5,5825ce2e88d5346438686b0bba99e5ee,2017-08-16 13:09:20,149.90,149.90
4,000379cdec625522490c315e70c7a9fb,0ab7fb08086d4af9141453c91878ed7a,2018-04-02 13:42:17,93.00,93.00
5,0004164d20a9e969af783496f3408652,cd3558a10d854487b4f907e9b326a4fc,2017-04-12 08:35:12,59.99,59.99
6,000419c5494106c306a97b5635748086,07f6c3baf9ac86865b60f640c4f923c6,2018-03-02 17:47:40,34.30,34.30
7,00046a560d407e99b969756e0b10f282,8c3d752c5c02227878fae49aeaddbfd7,2017-12-18 11:08:30,120.90,120.90
8,00050bf6e01e69d5c0fd612f1bcfb69c,fa906f338cee30a984d0945b3832e431,2017-09-17 16:04:44,69.99,69.99
9,000598caf2ef4117407665ac33275130,9b961b894e797f63622137ff7eb1c1af,2018-08-11 12:14:35,1107.00,1107.00


## Q12. CALCULATE THE CUMULATIVE SALES PER MONTH FOR EACH YEAR

In [31]:
temp = orders.merge(order_items, on='order_id', how='inner')

temp['order_purchase_timestamp'] = pd.to_datetime(temp['order_purchase_timestamp'])

temp['year'] = temp['order_purchase_timestamp'].dt.year
temp['month'] = temp['order_purchase_timestamp'].dt.month
temp['month_name'] = temp['order_purchase_timestamp'].dt.month_name()
monthly_sales = temp.groupby(['year', 'month', 'month_name'])['price'].sum().reset_index()
monthly_sales = monthly_sales.sort_values(['year', 'month'])

monthly_sales['cumulative_sales'] = monthly_sales.groupby('year')['price'].cumsum()

monthly_sales.rename(columns={'price': 'monthly_sales'}, inplace=True)

monthly_sales = monthly_sales[
    ['year', 'month', 'month_name', 'monthly_sales', 'cumulative_sales']
]

monthly_sales = monthly_sales.reset_index(drop=True)

monthly_sales['monthly_sales'] = monthly_sales['monthly_sales'].round(6)
monthly_sales['cumulative_sales'] = monthly_sales['cumulative_sales'].round(6)
monthly_sales.head(10)

,year,month,month_name,monthly_sales,cumulative_sales
0,2016,9,September,267.36,267.36
1,2016,10,October,49507.66,49775.02
2,2016,12,December,10.90,49785.92
3,2017,1,January,120312.87,120312.87
4,2017,2,February,247303.02,367615.89
5,2017,3,March,374344.30,741960.19
6,2017,4,April,359927.23,1101887.42
7,2017,5,May,506071.14,1607958.56
8,2017,6,June,433038.60,2040997.16
9,2017,7,July,498031.48,2539028.64


## Q13. CALCULATE THE YEAR OVER YEAR GROWTH RATE OF TOTAL SALES

In [32]:
temp = orders.merge(order_items, on='order_id', how='inner')
temp['order_purchase_timestamp'] = pd.to_datetime(temp['order_purchase_timestamp'])
temp['year'] = temp['order_purchase_timestamp'].dt.year
yearly_sales = temp.groupby('year')['price'].sum().reset_index()
yearly_sales.rename(columns={'price': 'total_sales'}, inplace=True)

yearly_sales = yearly_sales.sort_values('year')

yearly_sales['previous_year_sales'] = yearly_sales['total_sales'].shift(1)

yearly_sales['yoy_growth_percentage'] = (
    (yearly_sales['total_sales'] - yearly_sales['previous_year_sales']) /
    yearly_sales['previous_year_sales']
) * 100

yearly_sales['total_sales'] = yearly_sales['total_sales'].round(6)
yearly_sales['previous_year_sales'] = yearly_sales['previous_year_sales'].round(6)
yearly_sales['yoy_growth_percentage'] = yearly_sales['yoy_growth_percentage'].round(2)

yearly_sales = yearly_sales.reset_index(drop=True)
yearly_sales

,year,total_sales,previous_year_sales,yoy_growth_percentage
0,2016,49785.92,NaN,NaN
1,2017,6155806.98,49785.92,12264.55
2,2018,7386050.80,6155806.98,19.99


## Q14. CALCULATE THE RETENTION RATE OF CUSTOMERS

In [33]:
# Ensure datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Sort orders
orders_sorted = orders.sort_values(['customer_id','order_purchase_timestamp'])

# First purchase
first_order = orders_sorted.groupby('customer_id')['order_purchase_timestamp'].first().reset_index()

# Merge
orders_merged = orders_sorted.merge(first_order, on='customer_id', suffixes=('','_first'))

# Days difference
orders_merged['diff_days'] = (orders_merged['order_purchase_timestamp'] -
                              orders_merged['order_purchase_timestamp_first']).dt.days

# Retained customers (within 180 days)
retained_customers = orders_merged[
    (orders_merged['diff_days'] > 0) & (orders_merged['diff_days'] <= 180)
]['customer_id'].nunique()

total_customers = orders['customer_id'].nunique()

retention_rate = (retained_customers / total_customers) * 100

retention_rate

0.0

## Q15. IDENTIFY THE TOP 3 CUSTOMERS WHO SPENT THE MOST MONEY IN EACH YEAR

In [34]:
temp = orders.merge(order_items, on='order_id', how='inner')
temp['order_purchase_timestamp'] = pd.to_datetime(temp['order_purchase_timestamp'])

temp['year'] = temp['order_purchase_timestamp'].dt.year
result = temp.groupby(['year', 'customer_id'])['price'].sum().reset_index()
result.rename(columns={'price': 'total_spent'}, inplace=True)
result['rnk'] = result.groupby('year')['total_spent'] \
    .rank(method='min', ascending=False).astype(int)

result = result[result['rnk'] <= 3]
result = result[['year', 'customer_id', 'total_spent', 'rnk']]
result = result.sort_values(['year', 'rnk', 'customer_id'])
result = result.reset_index(drop=True)
result

,year,customer_id,total_spent,rnk
0,2016,a9dc96b027d1252bbac0a9b72d837fc6,1399.00,1
1,2016,1d34ed25963d5aae4cf3d7f3a4cda173,1299.99,2
2,2016,4a06381959b6670756de02e07b83815f,1199.00,3
3,2017,1617b1357756262bfa56ab541c47bc16,13440.00,1
4,2017,c6e2731c5b391845f6800c97401a43a9,6735.00,2
5,2017,3fd6777bbce08a352fddd04e4a7cc8f6,6499.00,3
6,2018,ec5b2ba62e574342386871631fafd3fc,7160.00,1
7,2018,f48d464a0baaea338cb25f816991ab1f,6729.00,2
8,2018,e0a2412720e9ea4f26c1ac985f6a7358,4599.90,3
